# Final match dataset builder

This notebook creates the match-level dataset used later for modelling.

It combines three data sources:
1. historical match results from `results.csv`,
2. yearly FIFA rankings,
3. Transfermarkt squad age and market value data.

The final table keeps only relevant matches from 2022 onward, adds ranking/value/form variables, and saves `df_fin_v1.csv`.



## Setup

Imports the libraries used for file paths, text cleaning and data handling.

In [ ]:
# Load basic libraries used in the notebook.
from pathlib import Path
import re
import unicodedata

import pandas as pd

## Configuration

Defines input files, output files, excluded tournaments and country-name fixes used before merging datasets.

In [ ]:
# Set all file names and constants in one place.
RESULTS_FILE = "results.csv"
TRANSFERMARKT_FILE = "transfermarkt_national_teams.csv"
OUTPUT_FILE = "df_fin_v1.csv"
COUNTRY_CHECK_FILE = "country_report.csv"

MIN_YEAR = 2022

RANKING_FILES = {
    2022: "fifa_rankings_2022.csv",
    2023: "fifa_rankings_2023.csv",
    2024: "fifa_rankings_2024.csv",
    2025: "fifa_rankings_2025.csv",
    2026: "fifa_rankings_2026.csv",
}

EXCLUDED_TOURNAMENTS = [
    "Friendly",
    "CONIFA Africa Football Cup",
]

COUNTRY_NAME_MAP = {
    # Align names that differ between match results and FIFA rankings.
    "Brunei": "Brunei Darussalam",
    "Cape Verde": "Cabo Verde",
    "Czech Republic": "Czechia",
    "DR Congo": "Congo DR",
    "Democratic Republic of the Congo": "Congo DR",
    "Gambia": "The Gambia",
    "Hong Kong": "Hong Kong, China",
    "Iran": "IR Iran",
    "Ivory Coast": "Côte d'Ivoire",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Kyrgyzstan": "Kyrgyz Republic",
    "North Korea": "Korea DPR",
    "South Korea": "Korea Republic",
    "Taiwan": "Chinese Taipei",
    "United States": "USA",
    "United States of America": "USA",
    "Virgin Islands": "US Virgin Islands",
    "United States Virgin Islands": "US Virgin Islands",

    # Align names that differ between Transfermarkt and the other sources.
    "China PR": "China",
    "PR China": "China",
    "People's Republic of China": "China",
    "Mainland China": "China",
    "China": "China",
    "Türkiye": "Turkey",
    "Turkiye": "Turkey",
    "Turkey": "Turkey",

    # Additional Transfermarkt naming differences.
    "Antigua/Barbuda": "Antigua and Barbuda",
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Bosnia and Herz.": "Bosnia and Herzegovina",
    "Central Africa": "Central African Republic",
    "Central African Rep.": "Central African Republic",
    "Republic of the Congo": "Congo",
    "Curacao": "Curaçao",
    "East Timor": "Timor-Leste",
    "Guinea Bissau": "Guinea-Bissau",
    "Ireland": "Republic of Ireland",
    "Macao": "Macau",
    "Macedonia": "North Macedonia",
    "Moldavia": "Moldova",
    "Palestina": "Palestine",
    "St. Kitts/Nevis": "St Kitts and Nevis",
    "Saint Kitts and Nevis": "St Kitts and Nevis",
    "St. Kitts and Nevis": "St Kitts and Nevis",
    "St. Lucia": "St Lucia",
    "Saint Lucia": "St Lucia",
    "St. Vincent/Grenadines": "St Vincent and the Grenadines",
    "Saint Vincent and the Grenadines": "St Vincent and the Grenadines",
    "St. Vincent and the Grenadines": "St Vincent and the Grenadines",
    "Sao Tome and Principe": "São Tomé and Príncipe",
    "Sao Tome & Principe": "São Tomé and Príncipe",
    "São Tomé & Príncipe": "São Tomé and Príncipe",
    "Swaziland": "Eswatini",
    "Trinidad & Tobago": "Trinidad and Tobago",
    "Turks-Caicos Islands": "Turks and Caicos Islands",
    "UAE": "United Arab Emirates",
    "Viet Nam": "Vietnam",
}

VALUE_COLUMNS = [
    "home_avg_age",
    "home_market_value",
    "away_avg_age",
    "away_market_value",
]

## Country name normalizer

This helper makes country names comparable across results, FIFA rankings and Transfermarkt.

In [ ]:
# Helper class for cleaning and standardizing country names.
class CountryNameNormalizer:
    """Standardizes country names before merging datasets."""

    def __init__(self, mapping):
        self.mapping = mapping
        self.lookup = {self.clean_key(key): value for key, value in mapping.items()}

    @staticmethod
    def clean_text(value):
        # Removes extra spaces but keeps the readable country name.
        if pd.isna(value):
            return value

        value = str(value).strip()
        value = " ".join(value.split())
        return value

    @staticmethod
    def clean_key(value):
        # Creates a simplified key so similar names can be matched.
        if pd.isna(value):
            return value

        value = str(value).strip().lower()
        value = unicodedata.normalize("NFKD", value)
        value = "".join(char for char in value if not unicodedata.combining(char))
        value = value.replace("&", "and")
        value = re.sub(r"[^a-z0-9]+", " ", value)
        value = " ".join(value.split())
        return value

    def normalize(self, value):
        # Uses the mapping table when a known alternative name is found.
        value = self.clean_text(value)

        if pd.isna(value):
            return value

        return self.lookup.get(self.clean_key(value), value)

    def normalize_columns(self, df, columns):
        # Applies the same name normalization to selected dataframe columns.
        df = df.copy()

        for column in columns:
            if column in df.columns:
                df[column] = df[column].map(self.normalize)

        return df

## Dataset builder

This class loads the raw files, filters the matches, merges all variables and creates the final modelling dataset.

In [ ]:
# Main class that performs the complete data-building pipeline.
class MatchDatasetBuilder:
    """Builds the final match-level dataset from results, rankings and Transfermarkt data."""

    def __init__(
        self,
        results_file=RESULTS_FILE,
        transfermarkt_file=TRANSFERMARKT_FILE,
        ranking_files=RANKING_FILES,
        output_file=OUTPUT_FILE,
        country_check_file=COUNTRY_CHECK_FILE,
        min_year=MIN_YEAR,
        excluded_tournaments=EXCLUDED_TOURNAMENTS,
        country_name_map=COUNTRY_NAME_MAP,
    ):
        self.results_file = Path(results_file)
        self.transfermarkt_file = Path(transfermarkt_file)
        self.ranking_files = {year: Path(file) for year, file in ranking_files.items()}
        self.output_file = Path(output_file)
        self.country_check_file = Path(country_check_file)
        self.min_year = min_year
        self.excluded_tournaments = excluded_tournaments
        self.normalizer = CountryNameNormalizer(country_name_map)

        self.df_matches_ranked = None
        self.df_transfermarkt = None
        self.df_final = None

    @staticmethod
    def require_file(path):
        if not path.exists():
            raise FileNotFoundError(f"Missing file: {path}")

    @staticmethod
    def unique_teams(df):
        teams = pd.concat([df["home_team"], df["away_team"]]).dropna()
        return sorted(teams.unique())

    def load_matches(self):
        # Loads historical results and standardizes team/country names.
        self.require_file(self.results_file)

        df = pd.read_csv(self.results_file)
        df["date"] = pd.to_datetime(df["date"])
        df = self.normalizer.normalize_columns(df, ["home_team", "away_team", "country"])

        return df

    def load_rankings(self):
        # Loads yearly FIFA rankings and stacks them into one table.
        frames = []

        for year, path in self.ranking_files.items():
            self.require_file(path)

            df = pd.read_csv(path)
            df = df[["rank", "team"]].copy()
            df["team"] = df["team"].map(self.normalizer.normalize)
            df["year"] = year
            frames.append(df)

        df = pd.concat(frames, ignore_index=True)
        df["rank"] = pd.to_numeric(df["rank"], errors="coerce")
        df["year"] = df["year"].astype(int)
        df = df.dropna(subset=["rank", "team"])
        df["rank"] = df["rank"].astype(int)

        return df.drop_duplicates(subset=["team", "year"])

    def load_transfermarkt(self):
        # Loads Transfermarkt team age and market value variables.
        self.require_file(self.transfermarkt_file)

        df = pd.read_csv(self.transfermarkt_file)
        df = df.rename(columns={"country": "team", "Country": "team"})
        df["team"] = df["team"].map(self.normalizer.normalize)
        df["avg_age"] = pd.to_numeric(df["avg_age"], errors="coerce")
        df["market_value_eur"] = pd.to_numeric(df["market_value_eur"], errors="coerce")

        df = df.dropna(subset=["team"])
        df = df.sort_values("market_value_eur", ascending=False, na_position="last")

        return df[["team", "avg_age", "market_value_eur"]].drop_duplicates(subset=["team"])

    def align_transfermarkt_to_matches(self, df_transfermarkt, df_matches):
        # Aligns Transfermarkt team names with names used in the match dataset.
        df = df_transfermarkt.copy()
        match_teams = self.unique_teams(df_matches)
        match_lookup = {self.normalizer.clean_key(team): team for team in match_teams}

        df["team_key"] = df["team"].map(self.normalizer.clean_key)
        df["team"] = df["team_key"].map(match_lookup).fillna(df["team"])
        df = df.drop(columns=["team_key"])
        df = df.sort_values("market_value_eur", ascending=False, na_position="last")

        return df.drop_duplicates(subset=["team"])

    def clean_matches(self, df_matches, fifa_teams):
        # Keeps relevant matches from the selected period and removes excluded tournaments.
        df = df_matches.copy()
        df = df[df["date"].dt.year >= self.min_year]
        df = df[~df["tournament"].isin(self.excluded_tournaments)]
        df = df[df["home_team"].isin(fifa_teams) & df["away_team"].isin(fifa_teams)]

        return df.reset_index(drop=True)

    @staticmethod
    def add_rankings(df_matches, df_rankings):
        # Adds home and away FIFA ranking for the year of each match.
        df = df_matches.copy()
        df["year"] = df["date"].dt.year.astype(int)

        home_rankings = df_rankings.rename(columns={"team": "home_team", "rank": "home_rank"})
        away_rankings = df_rankings.rename(columns={"team": "away_team", "rank": "away_rank"})

        df = df.merge(home_rankings[["home_team", "year", "home_rank"]], on=["home_team", "year"], how="left")
        df = df.merge(away_rankings[["away_team", "year", "away_rank"]], on=["away_team", "year"], how="left")
        df["rank_difference"] = df["home_rank"] - df["away_rank"]
        df = df.dropna(subset=["home_rank", "away_rank"])
        df[["home_rank", "away_rank"]] = df[["home_rank", "away_rank"]].astype(int)

        return df.reset_index(drop=True)

    @staticmethod
    def add_transfermarkt(df_matches, df_transfermarkt):
        # Adds age and market value variables for both teams.
        df = df_matches.copy()

        home_values = df_transfermarkt.rename(
            columns={
                "team": "home_team",
                "avg_age": "home_avg_age",
                "market_value_eur": "home_market_value",
            }
        )
        away_values = df_transfermarkt.rename(
            columns={
                "team": "away_team",
                "avg_age": "away_avg_age",
                "market_value_eur": "away_market_value",
            }
        )

        df = df.merge(home_values[["home_team", "home_avg_age", "home_market_value"]], on="home_team", how="left")
        df = df.merge(away_values[["away_team", "away_avg_age", "away_market_value"]], on="away_team", how="left")
        df["age_difference"] = df["home_avg_age"] - df["away_avg_age"]
        df["value_difference"] = df["home_market_value"] - df["away_market_value"]

        return df.reset_index(drop=True)


    @staticmethod
    def match_result(home_score, away_score):
        # Encodes the match result: home win = 1, draw = 0, away win = 2.
        if pd.isna(home_score) or pd.isna(away_score):
            return pd.NA

        if home_score > away_score:
            return 1

        if home_score == away_score:
            return 0

        return 2

    @staticmethod
    def result_points(result):
        # Converts the result label into football points for form calculation.
        if pd.isna(result):
            return None, None

        if result == 1:
            return 3, 0

        if result == 0:
            return 1, 1

        if result == 2:
            return 0, 3

        return None, None

    @classmethod
    def add_results_and_form(cls, df_matches):
        # Adds result labels and form based on points from the previous five matches.
        df = df_matches.copy()
        df["home_score"] = pd.to_numeric(df["home_score"], errors="coerce")
        df["away_score"] = pd.to_numeric(df["away_score"], errors="coerce")
        df["result"] = [
            cls.match_result(home_score, away_score)
            for home_score, away_score in zip(df["home_score"], df["away_score"])
        ]

        df["home_form"] = 0
        df["away_form"] = 0

        form_history = {}
        ordered_index = df.sort_values(["date"]).index

        for index in ordered_index:
            row = df.loc[index]
            home_team = row["home_team"]
            away_team = row["away_team"]

            home_history = form_history.get(home_team, [])
            away_history = form_history.get(away_team, [])

            df.at[index, "home_form"] = sum(home_history[-5:])
            df.at[index, "away_form"] = sum(away_history[-5:])

            home_points, away_points = cls.result_points(row["result"])

            if home_points is not None and away_points is not None:
                form_history.setdefault(home_team, []).append(home_points)
                form_history.setdefault(away_team, []).append(away_points)

        df["form_difference"] = df["home_form"] - df["away_form"]

        return df.reset_index(drop=True)

    def build(self):
        # Runs the whole pipeline in the correct order.
        df_matches_raw = self.load_matches()
        df_rankings = self.load_rankings()
        df_transfermarkt = self.load_transfermarkt()

        fifa_teams = set(df_rankings["team"].unique())
        df_matches = self.clean_matches(df_matches_raw, fifa_teams)
        df_matches = self.add_rankings(df_matches, df_rankings)
        df_transfermarkt = self.align_transfermarkt_to_matches(df_transfermarkt, df_matches)
        df_final = self.add_transfermarkt(df_matches, df_transfermarkt)
        df_final = self.add_results_and_form(df_final)

        self.df_matches_ranked = df_matches
        self.df_transfermarkt = df_transfermarkt
        self.df_final = df_final

        return df_final

    def country_report(self):
        # Creates a check table for teams not matched correctly across sources.
        if self.df_final is None:
            self.build()

        match_teams = set(self.unique_teams(self.df_matches_ranked))
        tm_teams = set(self.df_transfermarkt["team"].dropna().unique())

        missing_home = self.df_final[self.df_final["home_market_value"].isna()]
        missing_away = self.df_final[self.df_final["away_market_value"].isna()]
        missing_teams = sorted(set(missing_home["home_team"]).union(set(missing_away["away_team"])))

        rows = []

        for team in missing_teams:
            rows.append({
                "team": team,
                "problem": "missing_in_transfermarkt",
                "home_rows": int((self.df_matches_ranked["home_team"] == team).sum()),
                "away_rows": int((self.df_matches_ranked["away_team"] == team).sum()),
            })

        for team in sorted(tm_teams - match_teams):
            rows.append({
                "team": team,
                "problem": "not_used_in_matches",
                "home_rows": 0,
                "away_rows": 0,
            })

        return pd.DataFrame(rows)

    def save(self, df):
        # Saves the final dataset as CSV.
        df.to_csv(self.output_file, index=False, encoding="utf-8-sig")
        return self.output_file

    def save_country_report(self):
        # Saves the country-name quality check.
        report = self.country_report()
        report.to_csv(self.country_check_file, index=False, encoding="utf-8-sig")
        return self.country_check_file

    def normalize_existing_csv(self, input_file, output_file):
        # Optionally normalizes older CSV files for comparison.
        input_path = Path(input_file)

        if not input_path.exists():
            return None

        df = pd.read_csv(input_path)
        df = self.normalizer.normalize_columns(df, ["home_team", "away_team", "country"])
        df.to_csv(output_file, index=False, encoding="utf-8-sig")

        return Path(output_file)

## Run builder

Runs the full pipeline, saves the final dataset and prints basic checks.

In [ ]:
# Build the final dataset and save all requested output files.
builder = MatchDatasetBuilder()

df_final = builder.build()
output_path = builder.save(df_final)
check_path = builder.save_country_report()

fresh_normalized = builder.normalize_existing_csv("df_fresh.csv", "df_fresh_normalized.csv")
final_normalized = builder.normalize_existing_csv("df_final.csv", "df_final_normalized.csv")

display(df_final.head())

print(f"Rows after FIFA ranking: {len(builder.df_matches_ranked)}")
print(f"Rows in final dataset: {len(df_final)}")
print(f"Saved final dataset: {output_path}")
print(f"Saved country check: {check_path}")

if fresh_normalized:
    print(f"Saved normalized fresh CSV: {fresh_normalized}")

if final_normalized:
    print(f"Saved normalized final CSV: {final_normalized}")

## Transfermarkt check

Checks whether any teams still have missing Transfermarkt values after name normalization.

In [ ]:
# Check whether Transfermarkt variables were merged successfully.
missing_home = df_final[df_final["home_market_value"].isna()]["home_team"].value_counts()
missing_away = df_final[df_final["away_market_value"].isna()]["away_team"].value_counts()
missing_teams = sorted(set(missing_home.index).union(set(missing_away.index)))

print(f"Missing home Transfermarkt rows: {int(missing_home.sum())}")
print(f"Missing away Transfermarkt rows: {int(missing_away.sum())}")
print(f"Teams still missing in Transfermarkt: {len(missing_teams)}")

if missing_teams:
    display(pd.DataFrame({"team": missing_teams}))
else:
    print("No missing Transfermarkt team names.")

## Row check

Compares normalized older files with the new output when those files are available.

In [ ]:
# Compare older normalized files with the new dataset if both files exist.
fresh_path = Path("df_fresh_normalized.csv")
final_path = Path("df_final_normalized.csv")

if fresh_path.exists() and final_path.exists():
    df_fresh_check = pd.read_csv(fresh_path)
    df_final_check = pd.read_csv(final_path)

    match_columns = [
        "date", "home_team", "away_team", "home_score", "away_score",
        "tournament", "city", "country", "neutral", "year",
        "home_rank", "away_rank", "rank_difference",
    ]

    shared_columns = [column for column in match_columns if column in df_fresh_check.columns and column in df_final_check.columns]
    row_check = df_fresh_check[shared_columns].merge(
        df_final_check[shared_columns],
        on=shared_columns,
        how="outer",
        indicator=True,
    )

    print(row_check["_merge"].value_counts())
else:
    print("Normalized CSV files were not found in this folder.")

## Final preview

Loads the saved CSV and shows the first rows as a final check.

In [ ]:
# Open the saved dataset to verify that the file was created correctly.
df_check = pd.read_csv(OUTPUT_FILE)
df_check.head()